In [111]:
print("here starts the maincore file")

here starts the maincore file


In [3]:
# Run once — comment out after first install
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "spacy", "emoji", "scikit-learn", "pandas", "numpy", "joblib"
], check=True)
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 22.1 MB/s eta 0:00:0000:010:01



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


CompletedProcess(args=['/Users/yatharthnehva/.pyenv/versions/3.11.8/bin/python', '-m', 'spacy', 'download', 'en_core_web_sm'], returncode=0)

In [4]:
import re
import glob
import random
import warnings
import numpy as np
import pandas as pd
import joblib
import spacy
import emoji

from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

warnings.filterwarnings('ignore')

# ── Adjust these to match your directory layout ──────────────────────────────
# This notebook lives in: characterlevelattacks/coreattacks/
ROOT = Path("../")   # characterlevelattacks/

TWITTER_CSV  = Path("/Users/yatharthnehva/NLPproject/characterlevelattacks/emojibased/twittersenti/twittersenti140.csv")
SPGC_DIR     = ROOT / "stylometric" / "SPGC"
BLOG_DIR     = ROOT / "stylometric" / "Blogdataset"
MODEL_SAVE   = Path("register_clf.joblib")   # saved alongside this notebook
# ─────────────────────────────────────────────────────────────────────────────

print("Twitter CSV exists:", TWITTER_CSV.exists())
print("SPGC dir exists:", SPGC_DIR.exists())
print("Blog dir exists:", BLOG_DIR.exists())

Twitter CSV exists: True
SPGC dir exists: True
Blog dir exists: True


In [5]:
class ZoneDetector:
    """
    Builds a character-level protection mask for an input string.
    Protected positions must not be perturbed by any downstream attack.
    """

    URL_RE = re.compile(r'https?://\S+|www\.\S+')

    def __init__(self, spacy_model: str = "en_core_web_sm"):
        self.nlp = spacy.load(spacy_model)

    # ------------------------------------------------------------------ #

    def detect(self, text: str) -> dict:
        """
        Parameters
        ----------
        text : str

        Returns
        -------
        dict with keys:
            char_mask       : np.ndarray[bool], shape=(len(text),)
                              True  → position is protected
                              False → position is eligible for perturbation
            token_eligible  : list[bool], one entry per spaCy token
            tokens          : list[str]
            protected_spans : list[tuple(start, end, reason)]
        """
        n = len(text)
        mask = np.zeros(n, dtype=bool)
        spans = []

        # ── 1. URLs ──────────────────────────────────────────────────── #
        for m in self.URL_RE.finditer(text):
            mask[m.start():m.end()] = True
            spans.append((m.start(), m.end(), 'url'))

        # ── 2. Existing emoji ────────────────────────────────────────── #
        for i, ch in enumerate(text):
            if emoji.is_emoji(ch):
                mask[i] = True
                spans.append((i, i + 1, 'emoji'))

        # ── 3. Named entities + proper nouns (spaCy) ─────────────────── #
        doc = self.nlp(text)

        for ent in doc.ents:
            mask[ent.start_char:ent.end_char] = True
            spans.append((ent.start_char, ent.end_char, f'ent:{ent.label_}'))

        for tok in doc:
            if tok.pos_ == 'PROPN':
                mask[tok.idx: tok.idx + len(tok.text)] = True
                spans.append((tok.idx, tok.idx + len(tok.text), 'propn'))

        # ── 4. Sentence-final punctuation ────────────────────────────── #
        for sent in doc.sents:
            last = sent[-1]
            if last.is_punct:
                mask[last.idx: last.idx + len(last.text)] = True
                spans.append((last.idx, last.idx + len(last.text), 'sent_punct'))

        # ── Build per-token eligibility ──────────────────────────────── #
        token_eligible = [
            not mask[tok.idx: tok.idx + len(tok.text)].any()
            for tok in doc
        ]

        return {
            'char_mask':       mask,
            'token_eligible':  token_eligible,
            'tokens':          [tok.text for tok in doc],
            'protected_spans': spans,
        }

    # ── Pretty-print helper ─────────────────────────────────────────── #

    def visualise(self, text: str) -> None:
        """Print text with protected characters marked in brackets."""
        result = self.detect(text)
        mask = result['char_mask']
        out, in_block = [], False
        for i, ch in enumerate(text):
            if mask[i] and not in_block:
                out.append('['); in_block = True
            elif not mask[i] and in_block:
                out.append(']'); in_block = False
            out.append(ch)
        if in_block:
            out.append(']')
        print(''.join(out))
        eligible = sum(result['token_eligible'])
        total    = len(result['token_eligible'])
        print(f"  → {eligible}/{total} tokens eligible for perturbation")
        print(f"  → Protected spans: {result['protected_spans']}\n")

In [6]:
# Testing the detector

zd = ZoneDetector()

tests = [
    "The quick brown fox jumps over the lazy dog.",
    "Check out https://github.com/Ynehra24/NLPproject for the code!",
    "Elon Musk announced Tesla's new model yesterday 😊.",
    "The Battle of Waterloo changed European history forever.",
    "I love this so much lol it's hilarious",
]

for t in tests:
    zd.visualise(t)

The quick brown fox jumps over the lazy dog[.]
  → 9/10 tokens eligible for perturbation
  → Protected spans: [(43, 44, 'sent_punct')]

Check out [https://github.com/Ynehra24/NLPproject] for the code[!]
  → 5/7 tokens eligible for perturbation
  → Protected spans: [(10, 48, 'url'), (61, 62, 'sent_punct')]

[Elon Musk] announced [Tesla]'s new model [yesterday] [😊.]
  → 4/10 tokens eligible for perturbation
  → Protected spans: [(48, 49, 'emoji'), (0, 9, 'ent:PERSON'), (20, 25, 'ent:ORG'), (38, 47, 'ent:DATE'), (0, 4, 'propn'), (5, 9, 'propn'), (20, 25, 'propn'), (49, 50, 'sent_punct')]

[The Battle of Waterloo] changed [European] history forever[.]
  → 3/9 tokens eligible for perturbation
  → Protected spans: [(0, 22, 'ent:WORK_OF_ART'), (31, 39, 'ent:NORP'), (4, 10, 'propn'), (14, 22, 'propn'), (55, 56, 'sent_punct')]

I love this so much lol it's hilarious
  → 9/9 tokens eligible for perturbation
  → Protected spans: []



In [7]:
# ── Data loaders ─────────────────────────────────────────────────────────── #

def load_twitter(path: Path, n: int = 15_000, seed: int = 42) -> list[str]:
    """
    Sentiment140 CSV format:
    target | id | date | flag | user | text
    Returns a sample of `n` tweet texts.
    """
    df = pd.read_csv(
        path, encoding='latin-1', header=None,
        names=['target', 'id', 'date', 'flag', 'user', 'text']
    )
    texts = df['text'].dropna().tolist()
    # Filter very short or empty strings
    texts = [t for t in texts if isinstance(t, str) and len(t.strip()) > 20]
    random.seed(seed)
    return random.sample(texts, min(n, len(texts)))


def load_spgc(path: Path, n: int = 15_000, seed: int = 42) -> list[str]:
    """
    Loads formal sentences from SPGC (Project Gutenberg) Parquet file.
    """
    df = pd.read_parquet(path)
    
    # Try to find the text column automatically
    col = next((c for c in df.columns
                if any(k in c.lower() for k in ('text', 'content', 'body'))),
               None)
    
    if col:
        texts = df[col].dropna().astype(str).tolist()
    else:
        # Fallback to the first column if typical names aren't found
        texts = df.iloc[:, 0].dropna().astype(str).tolist()

    texts = [t for t in texts if len(t.strip()) > 30]
    random.seed(seed)
    return random.sample(texts, min(n, len(texts)))


# Quick preview
print("Loading data...")
# Assuming TWITTER_CSV is defined earlier in your notebook
informal_texts = load_twitter(TWITTER_CSV) 

SPGC_PARQUET = Path("/Users/yatharthnehva/NLPproject/characterlevelattacks/stylometric/SPGC/gutenberg_en_20k.parquet")
formal_texts   = load_spgc(SPGC_PARQUET)

print(f"Informal samples : {len(informal_texts):,}")
print(f"Formal samples   : {len(formal_texts):,}")
print("\nInformal sample:", informal_texts[0])
print("Formal sample:  ", formal_texts[0])


Loading data...
Informal samples : 15,000
Formal samples   : 15,000

Informal sample: Wow! 25 years of #Tetris! Thank you Alexey for making this great puzzle game!  http://bit.ly/11icWS
Formal sample:   The Project Gutenberg eBook, Arthur Machen, by Vincent Starrett


This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy it, give it away or
re-use it under the terms of the Project Gutenberg License included
with this eBook or online at www.gutenberg.org





Title: Arthur Machen
       A Novelist of Ecstasy and Sin


Author: Vincent Starrett



Release Date: March 7, 2011  [eBook #35515]

Language: English

Character set encoding: ISO-8859-1


***START OF THE PROJECT GUTENBERG EBOOK ARTHUR MACHEN***


E-text prepared by Marc D'Hooghe (http://www.freeliterature.org) from page
images generously made available by Internet Archive
(http://www.archive.org)



Note: Images of the original pages are available through
      Internet A

In [40]:
import joblib
from pathlib import Path
from tqdm import tqdm
import numpy as np
import time
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

MODEL_SAVE = Path("register_clf.joblib")


class RegisterClassifier:

    def __init__(self):
        self._init_vectorizer()
        self._init_model()
        self.trained = False

    # ───────────────────────────────────────────── #
    # CLEAN TEXT (SAFE)
    # ───────────────────────────────────────────── #

    def clean_text(self, text):
        text = text.lower()

        # remove obvious Gutenberg boilerplate
        if "project gutenberg" in text:
            return ""
        if "this ebook is for the use" in text:
            return ""

        # light cleanup only
        text = re.sub(r'\s+', ' ', text).strip()

        return text

    # ───────────────────────────────────────────── #
    # SAFE FORMAL FILTER (DO NOT OVER-FILTER)
    # ───────────────────────────────────────────── #

    def filter_formal(self, texts):
        good = []

        print("\n🔍 Cleaning formal texts...")

        for t in tqdm(texts, desc="Formal Clean"):
            t = self.clean_text(t)

            # keep almost everything meaningful
            if len(t) < 30:
                continue

            if "ebook" in t or "gutenberg" in t:
                continue

            good.append(t)

        return good

    # ───────────────────────────────────────────── #
    # VECTORIZER
    # ───────────────────────────────────────────── #

    def _init_vectorizer(self):
        print("🔧 Initializing vectorizer...")

        self.vec = TfidfVectorizer(
            ngram_range=(1, 2),
            max_features=20000,
            stop_words='english'
        )

    # ───────────────────────────────────────────── #
    # MODEL
    # ───────────────────────────────────────────── #

    def _init_model(self):
        print("🧠 Initializing model...")

        self.clf = SGDClassifier(
            loss='log_loss',
            random_state=42
        )

    # ───────────────────────────────────────────── #
    # VECTORIZE
    # ───────────────────────────────────────────── #

    def vectorize(self, X_train, X_test):
        print("\n🔤 TF-IDF vectorizing...")
        t0 = time.time()

        X_train_vec = self.vec.fit_transform(X_train)
        X_test_vec = self.vec.transform(X_test)

        print(f"   Done in {time.time() - t0:.2f}s")
        print(f"   Shape: {X_train_vec.shape}")

        return X_train_vec, X_test_vec

    # ───────────────────────────────────────────── #
    # TRAIN MODEL
    # ───────────────────────────────────────────── #

    def train_model(self, X_train, y_train, X_test, y_test, epochs=5, batch_size=1024):

        print("\n🧠 Training model...")

        y_train = np.array(y_train)
        classes = np.unique(y_train)

        # 🚨 SAFETY CHECK
        if len(classes) < 2:
            raise ValueError("❌ Only one class found. Dataset issue.")

        # class weights
        class_weights = compute_class_weight(
            class_weight='balanced',
            classes=classes,
            y=y_train
        )
        class_weight_dict = dict(zip(classes, class_weights))

        for epoch in range(epochs):
            print(f"\n📘 Epoch {epoch+1}/{epochs}")

            indices = np.random.permutation(X_train.shape[0])
            X_train = X_train[indices]
            y_train = y_train[indices]

            losses = []

            for i in tqdm(range(0, X_train.shape[0], batch_size), desc="Training"):

                X_batch = X_train[i:i+batch_size]
                y_batch = y_train[i:i+batch_size]

                weights = np.array([class_weight_dict[y] for y in y_batch])

                self.clf.partial_fit(
                    X_batch,
                    y_batch,
                    classes=classes,
                    sample_weight=weights
                )

                # stable loss
                probs = self.clf.predict_proba(X_batch)
                probs = np.clip(probs, 1e-10, 1.0)

                idxs = [list(self.clf.classes_).index(y) for y in y_batch]
                loss = -np.mean(np.log(probs[np.arange(len(y_batch)), idxs]))
                losses.append(loss)

            # evaluation
            y_pred = self.clf.predict(X_test)
            acc = accuracy_score(y_test, y_pred)

            print(f"   📉 Loss: {np.mean(losses):.4f}")
            print(f"   🎯 Accuracy: {acc:.4f}")

    # ───────────────────────────────────────────── #
    # FULL TRAIN PIPELINE
    # ───────────────────────────────────────────── #

    def train(self, informal_texts, formal_texts):

        print("\n📦 Cleaning dataset...")

        print("\n🧹 Cleaning informal...")
        informal = [self.clean_text(t) for t in tqdm(informal_texts)]

        formal = self.filter_formal(formal_texts)

        print(f"\nAfter cleaning:")
        print(f"Informal: {len(informal)}")
        print(f"Formal: {len(formal)}")

        if len(formal) < 100:
            print("⚠️ Warning: formal dataset small, but continuing anyway...")

        X = informal + formal
        y = ['informal'] * len(informal) + ['formal'] * len(formal)

        # 🚨 FINAL SAFETY CHECK
        if len(set(y)) < 2:
            raise ValueError("❌ Still only one class after processing.")

        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=0.15,
            stratify=y,
            random_state=42
        )

        X_train = [t[:300] for t in X_train]
        X_test = [t[:300] for t in X_test]

        print(f"\n🚀 Training on {len(X_train):,} samples...")

        X_train_vec, X_test_vec = self.vectorize(X_train, X_test)

        self.train_model(X_train_vec, y_train, X_test_vec, y_test)

        print("\n📊 Final Evaluation:")
        print(classification_report(y_test, self.clf.predict(X_test_vec)))

        self.trained = True
        return self

    # ───────────────────────────────────────────── #
    # PREDICT (RULE + ML HYBRID)
    # ───────────────────────────────────────────── #

    def predict(self, text):
        if not self.trained:
            raise RuntimeError("Model not trained or loaded.")

        text_low = text.lower()

        # 🔥 rule-based boost
        formal_markers = [
            "therefore", "however", "significant",
            "hypothesis", "analysis", "methodology",
            "results indicate", "conclusion"
        ]

        if any(word in text_low for word in formal_markers):
            return "formal", 0.95

        vec = self.vec.transform([text[:300]])

        proba = self.clf.predict_proba(vec)[0]
        idx = proba.argmax()

        return self.clf.classes_[idx], float(proba[idx])

    # ───────────────────────────────────────────── #
    # SAVE / LOAD
    # ───────────────────────────────────────────── #

    def save(self, path=MODEL_SAVE):
        joblib.dump((self.vec, self.clf), path)
        print(f"💾 Saved to {path}")

    @classmethod
    def load(cls, path=MODEL_SAVE):
        obj = cls()
        obj.vec, obj.clf = joblib.load(path)
        obj.trained = True
        print(f"📂 Loaded from {path}")
        return obj

In [41]:
reg = RegisterClassifier()
reg.train(informal_texts, formal_texts)
reg.save()

🔧 Initializing vectorizer...
🧠 Initializing model...

📦 Cleaning dataset...

🧹 Cleaning informal...


100%|██████████| 15000/15000 [00:00<00:00, 124065.41it/s]



🔍 Cleaning formal texts...


Formal Clean: 100%|██████████| 15000/15000 [00:30<00:00, 498.80it/s]



After cleaning:
Informal: 15000
Formal: 0
⚠️ Warning: formal dataset small, but continuing anyway...


ValueError: ❌ Still only one class after processing.

In [35]:
# LOAD TRAINED MODEL
reg_clf = RegisterClassifier.load("register_clf.joblib")

sanity = [
    ("omg this is literally the best thing ever lol", 'informal'),
    ("just saw my fave band and i cannot stop smiling rn", 'informal'),

    ("I would like to express my sincere appreciation for your assistance.", 'formal'),
    ("The results indicate a significant improvement in performance.", 'formal'),
    ("Please ensure that all requirements are met prior to submission.", 'formal'),

    ("honestly this movie was mid but the vibes were immaculate", 'informal'),
]
print(f"{'Text':<65} {'Expected':<10} {'Predicted':<10} {'Conf':>6}")
print("-" * 95)

correct = 0
for text, expected in sanity:
    label, conf = reg_clf.predict(text)
    tick = '✓' if label == expected else '✗'
    if label == expected:
        correct += 1
    print(f"{tick} {text[:62]:<63} {expected:<10} {label:<10} {conf:>6.3f}")

print(f"\nAccuracy on sanity set: {correct}/{len(sanity)}")

🔧 Initializing vectorizers...
🧠 Initializing model...
📂 Loaded from register_clf.joblib
Text                                                              Expected   Predicted    Conf
-----------------------------------------------------------------------------------------------
✓ omg this is literally the best thing ever lol                   informal   informal    0.979
✓ just saw my fave band and i cannot stop smiling rn              informal   informal    0.992
✗ I would like to express my sincere appreciation for your assis  formal     informal    0.988
✗ The results indicate a significant improvement in performance.  formal     informal    0.965
✗ Please ensure that all requirements are met prior to submissio  formal     informal    0.981
✓ honestly this movie was mid but the vibes were immaculate       informal   informal    0.980

Accuracy on sanity set: 3/6


# Gutenberg is really bad so here's a better set of data to use

In [44]:
from datasets import load_dataset
import pandas as pd
import random

TARGET_PER_SOURCE = 6666
SEED = 42
random.seed(SEED)

# ───────────────────────────────────────────── #
# CLEAN FUNCTION
# ───────────────────────────────────────────── #
def clean_text(t):
    if not t:
        return ""
    t = t.replace("\n", " ").strip()
    if len(t) < 50:
        return ""
    return t


# ───────────────────────────────────────────── #
# GENERIC STREAM SAMPLER
# ───────────────────────────────────────────── #
def stream_sample(dataset, field, k):
    samples = []

    for x in dataset:
        t = clean_text(x[field])
        if t:
            samples.append(t)

        if len(samples) >= k:
            break

    return samples


# ───────────────────────────────────────────── #
# 1. C4
# ───────────────────────────────────────────── #
print("📥 Loading C4...")

c4 = load_dataset("allenai/c4", "en", split="train", streaming=True)
c4_sample = stream_sample(c4, "text", TARGET_PER_SOURCE)

print(f"✅ C4: {len(c4_sample)}")


# ───────────────────────────────────────────── #
# 2. FINEWEB (REPLACES WIKIPEDIA)
# ───────────────────────────────────────────── #
print("📥 Loading FineWeb...")

fineweb = load_dataset("HuggingFaceFW/fineweb", split="train", streaming=True)
fineweb_sample = stream_sample(fineweb, "text", TARGET_PER_SOURCE)

print(f"✅ FineWeb: {len(fineweb_sample)}")


# ───────────────────────────────────────────── #
# 3. CNN / DAILYMAIL
# ───────────────────────────────────────────── #
print("📥 Loading CNN/DailyMail...")

news = load_dataset("cnn_dailymail", "3.0.0", split="train", streaming=True)
news_sample = stream_sample(news, "article", TARGET_PER_SOURCE)

print(f"✅ News: {len(news_sample)}")


# ───────────────────────────────────────────── #
# COMBINE
# ───────────────────────────────────────────── #
print("📦 Combining datasets...")

texts = c4_sample + fineweb_sample + news_sample

sources = (
    ["c4"] * TARGET_PER_SOURCE +
    ["fineweb"] * TARGET_PER_SOURCE +
    ["news"] * TARGET_PER_SOURCE
)

labels = ["formal"] * len(texts)

df = pd.DataFrame({
    "text": texts,
    "label": labels,
    "source": sources
})


# ───────────────────────────────────────────── #
# SHUFFLE
# ───────────────────────────────────────────── #
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)


# ───────────────────────────────────────────── #
# SAVE AS PARQUET
# ───────────────────────────────────────────── #
output_path = "formal_20k.parquet"
df.to_parquet(output_path, index=False)

print(f"\n✅ Saved {len(df)} samples to {output_path}")
print(df.head())

📥 Loading C4...
✅ C4: 6666
📥 Loading FineWeb...
✅ FineWeb: 6666
📥 Loading CNN/DailyMail...
✅ News: 6666
📦 Combining datasets...

✅ Saved 19998 samples to formal_20k.parquet
                                                text   label   source
0  (CNN) -- Emergency crews called off a search i...  formal     news
1  The ‘start Trainer Of Kg’ : Redefines What Can...  formal       c4
2  Understanding how landscape factors, including...  formal       c4
3  If you want to really enjoy your fly fishing e...  formal  fineweb
4  For the seventh year, the Academy of Country M...  formal  fineweb


In [79]:
import pandas as pd
import re

df = pd.read_parquet("/Users/yatharthnehva/NLPproject/characterlevelattacks/coreattacks/formal_augmented.parquet")

def clean_sentence(s):
    s = re.sub(r'\s+', ' ', s)           # collapse whitespace
    s = re.sub(r'\(.*?\)', '', s)        # remove parenthetical citations
    s = s.strip()
    return s

df['text'] = df['text'].apply(clean_sentence)
df = df[df['text'].str.len() > 20]       # remove very short sentences
df.to_parquet("formal_20k_clean_final.parquet", index=False)

In [78]:
import os
import re
import requests
import tarfile
import pandas as pd
from pathlib import Path
from datasets import load_dataset
from tqdm import tqdm

# ---------------------------
# Paths
# ---------------------------

FORMAL_20K_PATH = Path("/Users/yatharthnehva/NLPproject/characterlevelattacks/coreattacks/formal_20k.parquet")
OUTPUT_PATH = Path("/Users/yatharthnehva/NLPproject/characterlevelattacks/coreattacks/formal_augmented.parquet")
ENRON_TGZ_PATH = Path("enron_mail.tgz")
ENRON_EXTRACT_PATH = Path("enron_mail")

# ---------------------------
# Cleaning Function
# ---------------------------

def clean_text(text):
    if not text or not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# ---------------------------
# Load Existing Formal 20K
# ---------------------------

print("Loading existing formal 20K...")
formal_df = pd.read_parquet(FORMAL_20K_PATH)
formal_df["text"] = formal_df["text"].apply(clean_text)
print(f"Original formal samples: {len(formal_df)}")

# ---------------------------
# 1) BUSINESS EMAIL DATASET (Hugging Face)
# ---------------------------

print("\nDownloading Business Email dataset from HuggingFace…")
business = load_dataset("wardacoder/business-email-dataset", split="train")

# Extract the actual email text from 'output'
print("Extracting email text from Business Email dataset…")
biz_emails = [example["output"] for example in business]

biz_df = pd.DataFrame({"text": biz_emails})
biz_df["text"] = biz_df["text"].apply(clean_text)
print(f"Business email samples loaded: {len(biz_df)}")

# ---------------------------
# 2) ENRON EMAIL DATASET
# ---------------------------

if not ENRON_EXTRACT_PATH.exists():
    print("\nDownloading Enron Email dataset…")
    enron_url = "https://www.cs.cmu.edu/~./enron/enron_mail_20110402.tgz"
    r = requests.get(enron_url, stream=True)
    with open(ENRON_TGZ_PATH, "wb") as f:
        for chunk in tqdm(r.iter_content(chunk_size=8192), desc="Downloading"):
            f.write(chunk)

    print("Extracting Enron dataset…")
    with tarfile.open(ENRON_TGZ_PATH, "r:gz") as tar:
        tar.extractall(ENRON_EXTRACT_PATH)

# Collect Enron email text files
print("\nReading Enron emails…")
enron_texts = []
for root, dirs, files in os.walk(ENRON_EXTRACT_PATH):
    for file in files:
        if file.endswith(".txt"):
            path = os.path.join(root, file)
            try:
                with open(path, encoding="latin1") as f:
                    enron_texts.append(f.read())
            except:
                continue

enron_df = pd.DataFrame({"text": enron_texts})
enron_df["text"] = enron_df["text"].apply(clean_text)
print(f"Total Enron email samples: {len(enron_df)}")

# ---------------------------
# 3) SAMPLE DATASETS
# ---------------------------

SAMPLE_SIZE = 6666

biz_sample = biz_df.sample(min(SAMPLE_SIZE, len(biz_df)), random_state=42)
enron_sample = enron_df.sample(min(SAMPLE_SIZE, len(enron_df)), random_state=42)

print(f"Sampled Business: {len(biz_sample)}, Enron: {len(enron_sample)}")

# ---------------------------
# 4) MERGE
# ---------------------------

combined_df = pd.concat([formal_df, biz_sample, enron_sample], ignore_index=True)
combined_df["label"] = "formal"

print(f"Total combined formal dataset: {len(combined_df)}")

# ---------------------------
# 5) SAVE
# ---------------------------

combined_df.to_parquet(OUTPUT_PATH, index=False)
print(f"\nSaved augmented formal dataset to:\n{OUTPUT_PATH}")

Loading existing formal 20K...
Original formal samples: 19998

Extracting email text from Business Email dataset…
Business email samples loaded: 10000



Downloading: 54135it [05:37, 160.32it/s]


Extracting Enron dataset…

Reading Enron emails…
Total Enron email samples: 1
Sampled Business: 6666, Enron: 1
Total combined formal dataset: 26665

Saved augmented formal dataset to:
/Users/yatharthnehva/NLPproject/characterlevelattacks/coreattacks/formal_augmented.parquet


In [80]:
# ---------------------------
# REGISTER CLASSIFIER: FIXED + STRONG FORMAL LEARNING
# ---------------------------

import joblib
import re
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.pipeline import FeatureUnion
from tqdm import tqdm

MODEL_SAVE = Path("register_clf_fixed_20k_each.joblib")

# ---------------------------
# CLASSIFIER
# ---------------------------
class RegisterClassifier:
    def __init__(self):
        self._init_vectorizer()
        self._init_model()
        self.trained = False

    # Aggressive cleaning but keeps formal cues
    def clean_text(self, text):
        if not text:
            return ""
        text = text.lower()
        text = re.sub(r"http\S+|www\S+", "", text)   # URLs
        text = re.sub(r"@\w+", "", text)             # mentions
        text = re.sub(r"#\w+", "", text)             # hashtags
        text = re.sub(r"[^\w\s]", " ", text)         # punctuation replaced by space (keep sentence separation)
        text = re.sub(r"\s+", " ", text)             # collapse spaces
        return text.strip()

    # Vectorizer with word + char ngrams (captures formal style patterns)
    def _init_vectorizer(self):
        word_vec = TfidfVectorizer(ngram_range=(1,2), max_features=20000, stop_words='english')
        char_vec = TfidfVectorizer(analyzer='char', ngram_range=(3,6), max_features=15000)
        self.vec = FeatureUnion([("word", word_vec), ("char", char_vec)])

    # SGDClassifier with log loss for probability outputs
    def _init_model(self):
        self.clf = SGDClassifier(loss='log_loss', random_state=42)

    def vectorize(self, X_train, X_test):
        X_train_vec = self.vec.fit_transform(X_train)
        X_test_vec = self.vec.transform(X_test)
        print(f"Vectorizer done. Shape: {X_train_vec.shape}")
        return X_train_vec, X_test_vec

    def train(self, informal_texts, formal_texts):
        # ---------------- CLEAN ----------------
        print("Cleaning formal texts...")
        formal_clean = [self.clean_text(t) for t in tqdm(formal_texts[:20000], desc="Formal")]
        print("Cleaning informal texts...")
        informal_clean = [self.clean_text(t) for t in tqdm(informal_texts[:20000], desc="Informal")]

        # Combine datasets
        X = formal_clean + informal_clean
        y = ['formal']*len(formal_clean) + ['informal']*len(informal_clean)

        # Truncate to 300 chars
        X = [t[:300] for t in X]

        # ---------------- SPLIT ----------------
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.15, stratify=y, random_state=42
        )

        # ---------------- VECTORIZATION ----------------
        X_train_vec, X_test_vec = self.vectorize(X_train, X_test)

        # ---------------- CLASS WEIGHTS ----------------
        classes = np.unique(y_train)
        class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
        class_weight_dict = dict(zip(classes, class_weights))
        sample_weights = np.array([class_weight_dict[label] for label in y_train])

        # ---------------- TRAIN ----------------
        print("Training classifier...")
        self.clf.fit(X_train_vec, y_train, sample_weight=sample_weights)

        # ---------------- EVALUATE ----------------
        y_pred = self.clf.predict(X_test_vec)
        print("\nFinal evaluation:")
        print(classification_report(y_test, y_pred))
        self.trained = True
        return self

    # Returns label + probability of the predicted class
    def predict(self, text):
        if not self.trained:
            raise RuntimeError("Model not trained")
        vec = self.vec.transform([self.clean_text(text)[:300]])
        proba = self.clf.predict_proba(vec)[0]
        idx = proba.argmax()
        return self.clf.classes_[idx], float(proba[idx])

    def save(self, path=MODEL_SAVE):
        joblib.dump((self.vec, self.clf), path)
        print(f"Saved model to {path}")

    @classmethod
    def load(cls, path=MODEL_SAVE):
        obj = cls()
        obj.vec, obj.clf = joblib.load(path)
        obj.trained = True
        print(f"Loaded model from {path}")
        return obj

# ---------------------------
# HELPER: Read CSV
# ---------------------------
def read_csv_autodetect(path):
    import chardet
    with open(path, "rb") as f:
        result = chardet.detect(f.read())
    print(f"Detected encoding: {result['encoding']}")
    df = pd.read_csv(path, encoding=result['encoding'], header=None)
    df.rename(columns={5:'text'}, inplace=True)  # Column 5 = tweet text
    return df

# ---------------------------
# LOAD DATA
# ---------------------------
df_formal = pd.read_parquet("/Users/yatharthnehva/NLPproject/characterlevelattacks/coreattacks/formal_20k_clean_final.parquet")
formal_texts = df_formal["text"].tolist()

df_informal = read_csv_autodetect("/Users/yatharthnehva/NLPproject/characterlevelattacks/emojibased/twittersenti/twittersenti140.csv")
informal_texts = df_informal["text"].tolist()

# ---------------------------
# TRAIN & SAVE
# ---------------------------
if __name__ == "__main__":
    reg = RegisterClassifier()
    reg.train(informal_texts, formal_texts)
    reg.save()

    # Example predictions
    test_texts = [
        "Please submit the report by Friday noon.",
        "Dear Professor, I would like to schedule a meeting.",
        "LOL that was hilarious 😂",
        "Are you free tomorrow?"
    ]
    for t in test_texts:
        label, prob = reg.predict(t)
        print(f"Text: {t}\nPredicted: {label} ({prob:.2f})\n")

Detected encoding: Windows-1252
Cleaning formal texts...


Formal: 100%|██████████| 20000/20000 [00:04<00:00, 4490.95it/s]


Cleaning informal texts...


Informal: 100%|██████████| 20000/20000 [00:00<00:00, 114970.87it/s]


Vectorizer done. Shape: (34000, 35000)
Training classifier...

Final evaluation:
              precision    recall  f1-score   support

      formal       0.99      0.98      0.99      3000
    informal       0.98      0.99      0.99      3000

    accuracy                           0.99      6000
   macro avg       0.99      0.99      0.99      6000
weighted avg       0.99      0.99      0.99      6000

Saved model to register_clf_fixed_20k_each.joblib
Text: Please submit the report by Friday noon.
Predicted: informal (0.80)

Text: Dear Professor, I would like to schedule a meeting.
Predicted: informal (0.94)

Text: LOL that was hilarious 😂
Predicted: informal (0.98)

Text: Are you free tomorrow?
Predicted: informal (0.98)



In [81]:
# ---------------------------
# TEST REGISTER CLASSIFIER
# ---------------------------

import joblib

# Load the trained model
reg = joblib.load("/Users/yatharthnehva/NLPproject/characterlevelattacks/coreattacks/register_clf_fixed_20k_each.joblib")
vec, clf = reg
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import SGDClassifier
from sklearn.feature_extraction.text import TfidfVectorizer

# Re-wrap in RegisterClassifier object for convenience
class TempRegister:
    def __init__(self, vec, clf):
        self.vec = vec
        self.clf = clf
        self.trained = True

    def predict(self, text):
        vec = self.vec.transform([text[:300]])
        proba = self.clf.predict_proba(vec)[0]
        idx = proba.argmax()
        return self.clf.classes_[idx], float(proba[idx])

reg = TempRegister(vec, clf)

# Sample texts
samples = [
    "Please submit the report by Friday noon.",
    "Dear Professor, I would like to schedule a meeting.",
    "The results of the experiment were conclusive."
]


# Test predictions
for text in samples:
    label, prob = reg.predict(text)
    print(f"Text: {text}\nPredicted: {label} (confidence: {prob:.2f})\n")

Text: Please submit the report by Friday noon.
Predicted: informal (confidence: 0.80)

Text: Dear Professor, I would like to schedule a meeting.
Predicted: informal (confidence: 0.94)

Text: The results of the experiment were conclusive.
Predicted: informal (confidence: 0.59)



# Even better methods/optimized methods don't work well for this task, so we should use a pre-trained model for it

In [83]:
pip install sentence_transformers


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [110]:
# ---------------------------
# FORMAL VS INFORMAL CLASSIFIER
# all-mpnet-base-v2 + stylistic features + full save
# ---------------------------

import re
import io
import joblib
from pathlib import Path
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import StandardScaler
import emoji

# ---------------------------
# Device
# ---------------------------

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"Using device: {device}")

# ---------------------------
# Paths
# ---------------------------

FORMAL_PATH   = Path("/Users/yatharthnehva/NLPproject/characterlevelattacks/coreattacks/formal_augmented.parquet").resolve()
INFORMAL_PATH = Path("/Users/yatharthnehva/NLPproject/characterlevelattacks/emojibased/twittersenti/twittersenti140.csv").resolve()

SAVE_DIR = Path("/Users/yatharthnehva/NLPproject/characterlevelattacks/coreattacks/formality_model")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

STYLE_WEIGHT = 5.0

# ---------------------------
# Cleaning
# ---------------------------

def clean_text(text):
    if not text or not isinstance(text, str):
        return ""
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# ---------------------------
# Stylistic features
# ---------------------------

def extract_style_features(texts):
    SLANG = {
        "lol", "lmao", "omg", "wtf", "tbh", "ngl", "idk", "imo", "brb",
        "gtg", "smh", "fyi", "asap", "gonna", "wanna", "gotta", "kinda",
        "sorta", "dunno", "lemme", "gimme", "ya", "yea", "yep", "nope",
        "dude", "bro", "sis", "lit", "vibe", "slay", "fr", "rn", "irl",
        "lowkey", "highkey", "bet", "fam", "squad", "thx", "ty",
        "np", "btw", "imho", "rofl", "lmfao", "af", "bc"
    }
    CONTRACTIONS = re.compile(
        r"\b(i'm|i've|i'd|i'll|you're|you've|you'd|you'll|he's|she's|"
        r"it's|we're|we've|we'd|we'll|they're|they've|they'd|they'll|"
        r"don't|doesn't|didn't|won't|wouldn't|can't|couldn't|shouldn't|"
        r"isn't|aren't|wasn't|weren't|hasn't|haven't|hadn't|"
        r"that's|there's|here's|what's|who's|how's|let's)\b", re.I
    )
    FORMAL_VOCAB = {
        "hereby", "pursuant", "accordingly", "therefore", "thus", "hence",
        "kindly", "sincerely", "regards", "respectfully", "dear", "attached",
        "herewith", "aforementioned", "notwithstanding", "henceforth",
        "request", "require", "submit", "review", "provide", "ensure",
        "deadline", "extension", "meeting", "schedule", "confirm", "inform",
        "apologize", "acknowledge", "appreciate", "assistance", "endeavor",
        "furthermore", "moreover", "however", "nevertheless", "consequently",
        "professor", "colleague", "committee", "department", "institution",
        "enclosed", "regarding", "subject", "proposal", "discuss", "concern"
    }

    features = []
    for text in texts:
        if not text or not isinstance(text, str):
            features.append([0] * 11)
            continue

        words   = text.split()
        n_words = max(len(words), 1)
        lower   = text.lower()

        n_emojis      = len([c for c in text if c in emoji.EMOJI_DATA]) / n_words
        n_exclaim     = text.count('!') / n_words
        n_question    = text.count('?') / n_words
        n_caps_words  = sum(1 for w in words if w.isupper() and len(w) > 1) / n_words
        n_contractions = len(CONTRACTIONS.findall(text)) / n_words
        word_lengths  = [len(re.sub(r'[^a-zA-Z]', '', w)) for w in words]
        avg_word_len  = np.mean(word_lengths) if word_lengths else 0
        word_tokens   = set(re.findall(r'\b\w+\b', lower))
        n_slang       = len(word_tokens & SLANG) / n_words
        n_repeated    = len(re.findall(r'(.)\1{2,}', text)) / n_words
        n_ellipsis    = text.count('...') / n_words
        formal_vocab_score = len(word_tokens & FORMAL_VOCAB) / n_words
        has_informal_interjection = int(bool(re.search(
            r'\b(lol|haha|hehe|omg|wow|yay|ugh|eww|woah|whoa|yikes|wtf)\b', lower
        )))

        features.append([
            n_emojis, n_exclaim, n_question, n_caps_words, n_contractions,
            avg_word_len, n_slang, n_repeated, n_ellipsis,
            formal_vocab_score, has_informal_interjection
        ])

    return np.array(features, dtype=np.float32)

# ---------------------------
# Load datasets
# ---------------------------

formal_df = pd.read_parquet(FORMAL_PATH)
formal_df["text"] = formal_df["text"].apply(clean_text)
formal_df = formal_df[formal_df["text"].str.len() > 5]
print(f"Formal samples: {len(formal_df)}")

cols = ["polarity", "id", "date", "query", "user", "text"]
with open(INFORMAL_PATH, "rb") as f:
    content = f.read().decode("utf-8", errors="replace")
informal_df = pd.read_csv(io.StringIO(content), names=cols, on_bad_lines='skip')
informal_df["text"] = informal_df["text"].apply(clean_text)
informal_df = informal_df[informal_df["text"].str.len() > 5]
print(f"Informal samples: {len(informal_df)}")

# ---------------------------
# Balance & split
# ---------------------------

n = min(len(formal_df), len(informal_df), 7000)
X = formal_df.sample(n=n, random_state=42)["text"].tolist() + \
    informal_df.sample(n=n, random_state=42)["text"].tolist()
y = ["formal"] * n + ["informal"] * n

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

# ---------------------------
# Encode with cache (auto-invalidates if data size changes)
# ---------------------------

def encode_with_cache(smodel, texts, cache_path, batch_size=64):
    if cache_path.exists():
        cached = np.load(cache_path)
        if cached.shape[0] == len(texts):
            print(f"Loaded cache: {cache_path.name}")
            return cached
        print(f"Cache size mismatch ({cached.shape[0]} vs {len(texts)}), re-encoding...")
    print(f"Encoding → {cache_path.name}")
    emb = smodel.encode(texts, show_progress_bar=True, convert_to_numpy=True, batch_size=batch_size)
    np.save(cache_path, emb)
    return emb

print("Loading all-mpnet-base-v2...")
embed_model = SentenceTransformer('all-mpnet-base-v2', device=device)

X_train_emb = encode_with_cache(embed_model, X_train, SAVE_DIR / "train_emb.npy")
X_test_emb  = encode_with_cache(embed_model, X_test,  SAVE_DIR / "test_emb.npy")

# ---------------------------
# Stylistic features
# ---------------------------

print("Extracting style features...")
X_train_style = extract_style_features(X_train)
X_test_style  = extract_style_features(X_test)

scaler = StandardScaler()
X_train_style = scaler.fit_transform(X_train_style)
X_test_style  = scaler.transform(X_test_style)

X_train_combined = np.concatenate([X_train_emb, X_train_style * STYLE_WEIGHT], axis=1)
X_test_combined  = np.concatenate([X_test_emb,  X_test_style  * STYLE_WEIGHT], axis=1)

# ---------------------------
# Train
# ---------------------------

classes = np.array(["formal", "informal"])
weights = compute_class_weight(class_weight='balanced', classes=classes, y=np.array(y_train))
class_weights = dict(zip(classes, weights))
print("Class weights:", class_weights)

clf = LogisticRegression(max_iter=2000, class_weight=class_weights, C=1.0)
clf.fit(X_train_combined, y_train)

# ---------------------------
# Evaluate
# ---------------------------

y_pred = clf.predict(X_test_combined)
print("\nEvaluation on test set:")
print(classification_report(y_test, y_pred))

# ---------------------------
# Save everything
# ---------------------------

embed_model.save(str(SAVE_DIR / "embed_model"))   # SentenceTransformer model weights
joblib.dump(clf,    SAVE_DIR / "classifier.joblib")
joblib.dump(scaler, SAVE_DIR / "scaler.joblib")

print(f"\nSaved to {SAVE_DIR}/")
print("  embed_model/       ← SentenceTransformer weights")
print("  classifier.joblib  ← logistic regression")
print("  scaler.joblib      ← StandardScaler")
print("  train_emb.npy      ← embedding cache")
print("  test_emb.npy       ← embedding cache")

# ---------------------------
# Predict function
# ---------------------------

def predict(text):
    text_clean = clean_text(text)
    emb        = embed_model.encode([text_clean], convert_to_numpy=True)
    style      = scaler.transform(extract_style_features([text_clean]))
    combined   = np.concatenate([emb, style * STYLE_WEIGHT], axis=1)
    pred       = clf.predict(combined)[0]
    proba      = clf.predict_proba(combined)[0].max()
    return pred, float(proba)

# ---------------------------
# Test examples
# ---------------------------

examples = [
    "Dear Professor, I would like to schedule a meeting.",
    "Please submit the report by Friday noon.",
    "I hereby request an extension on the deadline.",
    "Kindly find the attached report for your review.",
    "LOL that was hilarious 😂",
    "Are you free tomorrow?",
    "omg bro that was sooooo funny lmaooo 💀",
    "yo what's up are u coming tmrw??",
]

print()
for ex in examples:
    label, prob = predict(ex)
    print(f"{ex} => {label} ({prob:.2f})")


Using device: mps
Formal samples: 26665
Informal samples: 1592179
Train: 11900 | Test: 2100
Loading all-mpnet-base-v2...
Encoding → train_emb.npy


Batches: 100%|██████████| 186/186 [30:11<00:00,  9.74s/it] 


Encoding → test_emb.npy


Batches: 100%|██████████| 33/33 [05:03<00:00,  9.20s/it]


Extracting style features...
Class weights: {'formal': 1.0, 'informal': 1.0}

Evaluation on test set:
              precision    recall  f1-score   support

      formal       0.97      0.98      0.98      1050
    informal       0.98      0.97      0.98      1050

    accuracy                           0.98      2100
   macro avg       0.98      0.98      0.98      2100
weighted avg       0.98      0.98      0.98      2100


Saved to /Users/yatharthnehva/NLPproject/characterlevelattacks/coreattacks/formality_model/
  embed_model/       ← SentenceTransformer weights
  classifier.joblib  ← logistic regression
  scaler.joblib      ← StandardScaler
  train_emb.npy      ← embedding cache
  test_emb.npy       ← embedding cache

Dear Professor, I would like to schedule a meeting. => formal (0.86)
Please submit the report by Friday noon. => formal (0.98)
I hereby request an extension on the deadline. => formal (0.74)
Kindly find the attached report for your review. => formal (0.92)
LOL that w

# Now let us do this without any vocab lists as it is quite useless

In [ ]:
import re
import io
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report
from tqdm import tqdm

# ---------------------------
# Device
# ---------------------------

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"Using device: {device}")

# ---------------------------
# Load model directly as a classifier
# ---------------------------

MODEL_NAME = "s-nlp/roberta-base-formality-ranker"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
ranker     = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
ranker.eval()
print(f"Label map: {ranker.config.id2label}")

# ---------------------------
# Paths
# ---------------------------

FORMAL_PATH   = Path("/Users/yatharthnehva/NLPproject/characterlevelattacks/coreattacks/formal_augmented.parquet").resolve()
INFORMAL_PATH = Path("/Users/yatharthnehva/NLPproject/characterlevelattacks/emojibased/twittersenti/twittersenti140.csv").resolve()

# ---------------------------
# Cleaning
# ---------------------------

def clean_text(text):
    if not text or not isinstance(text, str):
        return ""
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# ---------------------------
# Predict (single text)
# ---------------------------

def predict(text):
    text_clean = clean_text(text)
    inputs = tokenizer(text_clean, return_tensors="pt", truncation=True,
                       max_length=128, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = ranker(**inputs).logits
    probs  = torch.softmax(logits, dim=-1)[0]
    pred   = probs.argmax().item()
    label  = ranker.config.id2label[pred]
    return label, probs[pred].item()

# ---------------------------
# Batched prediction (for evaluation)
# ---------------------------

def predict_batch(texts, batch_size=64):
    all_labels = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Evaluating"):
        batch = texts[i : i + batch_size]
        batch = [clean_text(t) for t in batch]
        enc = tokenizer(batch, return_tensors="pt", truncation=True,
                        max_length=128, padding=True)
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            logits = ranker(**enc).logits
        preds = logits.argmax(dim=-1).tolist()
        all_labels.extend([ranker.config.id2label[p] for p in preds])
    return all_labels

# ---------------------------
# Load test data for evaluation
# ---------------------------

formal_df = pd.read_parquet(FORMAL_PATH)
formal_df["text"] = formal_df["text"].apply(clean_text)
formal_df = formal_df[formal_df["text"].str.len() > 5]

cols = ["polarity", "id", "date", "query", "user", "text"]
with open(INFORMAL_PATH, "rb") as f:
    content = f.read().decode("utf-8", errors="replace")
informal_df = pd.read_csv(io.StringIO(content), names=cols, on_bad_lines='skip')
informal_df["text"] = informal_df["text"].apply(clean_text)
informal_df = informal_df[informal_df["text"].str.len() > 5]

n = min(len(formal_df), len(informal_df), 5000)  # 1000 per class is enough to evaluate
formal_sampled   = formal_df.sample(n=n, random_state=42)
informal_sampled = informal_df.sample(n=n, random_state=42)

X_test = formal_sampled["text"].tolist() + informal_sampled["text"].tolist()
y_test = ["formal"] * n + ["informal"] * n

# ---------------------------
# Evaluate
# ---------------------------

y_pred = predict_batch(X_test)
print("\nEvaluation on test set:")
print(classification_report(y_test, y_pred))

# ---------------------------
# Test examples
# ---------------------------

examples = [
    "Dear Professor, I would like to schedule a meeting.",
    "Please submit the report by Friday noon.",
    "I hereby request an extension on the deadline.",
    "Kindly find the attached report for your review.",
    "LOL that was hilarious 😂",
    "Are you free tomorrow?",
    "omg bro that was sooooo funny lmaooo 💀",
    "yo what's up are u coming tmrw??",
]

print()
for ex in examples:
    label, prob = predict(ex)
    print(f"{ex} => {label} ({prob:.2f})")


Using device: mps
Label map: {0: 'informal', 1: 'formal'}


Evaluating: 100%|██████████| 32/32 [01:58<00:00,  3.71s/it]



Evaluation on test set:
              precision    recall  f1-score   support

      formal       0.87      0.90      0.88      1000
    informal       0.89      0.87      0.88      1000

    accuracy                           0.88      2000
   macro avg       0.88      0.88      0.88      2000
weighted avg       0.88      0.88      0.88      2000


Dear Professor, I would like to schedule a meeting. => formal (1.00)
Please submit the report by Friday noon. => formal (0.86)
I hereby request an extension on the deadline. => formal (1.00)
Kindly find the attached report for your review. => formal (1.00)
LOL that was hilarious 😂 => informal (0.96)
Are you free tomorrow? => formal (0.95)
omg bro that was sooooo funny lmaooo 💀 => informal (0.99)
yo what's up are u coming tmrw?? => informal (1.00)


In [ ]:
informal_df.head()

,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer. You shoulda got David Carr of Third Day to do it. ;D"
0,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
1,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
2,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
3,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
4,0,1467811372,Mon Apr 06 22:20:00 PDT 2009,NO_QUERY,joy_wolf,@Kwesidei not the whole crew


In [62]:
print(df_informal.columns)

Index(['0', '1467810369', 'Mon Apr 06 22:19:45 PDT 2009', 'NO_QUERY',
       '_TheSpecialOne_',
       '@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer.  You shoulda got David Carr of Third Day to do it. ;D'],
      dtype='object')


In [33]:
print("Informal:", len(informal_texts))
print("Formal:", len(formal_texts))

Informal: 15000
Formal: 15000


In [34]:
for t in formal_texts[:5]:
    print("\n", t[:200])


 The Project Gutenberg eBook, Arthur Machen, by Vincent Starrett


This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy it, give it away or


 The Project Gutenberg eBook, Famous Authors (Men), by E. F. (Edward
Francis) Harkins


This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy

 ﻿Project Gutenberg's The Lenâpé and their Legends, by Daniel G. Brinton

This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy it, give it a

 The Project Gutenberg eBook, Fenton's Quest, by M. E. Braddon


This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy it, give it away or
re

 ﻿The Project Gutenberg EBook of Essays Literary, Critical and Historical, by 
Thomas O'Hagan

This eBook is for the use of anyone anywhere in the United States and most
other parts of th